Supply Chain & Inventory Analytics — Data Cleaning\nRaw CSVs → inspect → clean → export clean CSVs for MySQL load.

1. Supplier Data

In [3]:
import pandas as pd
import numpy as np

In [ ]:
suppliers = pd.read_csv("D:\\Data_Analysis\\SupplyChain and Inventory Analysis\\suppliers_raw.csv")

In [ ]:
suppliers.info()

In [ ]:
suppliers.describe()

In [ ]:
print(suppliers.isnull().sum())
print("Duplicates:", suppliers.duplicated().sum())
print(suppliers["country"].unique())
print(suppliers["lead_time_days"].describe())

In [ ]:
suppliers["country"] = suppliers["country"].str.strip().str.title()
 
suppliers.loc[suppliers["lead_time_days"] < 0, "lead_time_days"] = np.nan
suppliers["lead_time_days"] = suppliers["lead_time_days"].fillna(
    suppliers["lead_time_days"].median()
)
 
suppliers["reliability_score"] = suppliers["reliability_score"].fillna(
    suppliers["reliability_score"].median()
)

In [ ]:
assert suppliers["lead_time_days"].isnull().sum() == 0
assert suppliers["reliability_score"].isnull().sum() == 0
assert (suppliers["lead_time_days"] >= 0).all()

In [ ]:
suppliers.head()

In [ ]:
print(suppliers['country'].unique())

In [ ]:
suppliers.to_csv("suppliers_clean.csv", index=False)

In [ ]:
suppliers.head(10)

2. Warehouse Data

In [ ]:
warehouses = pd.read_csv("D:\\Data_Analysis\\SupplyChain and Inventory Analysis\\warehouses_raw.csv")

In [ ]:
warehouses.info()
warehouses.describe()

In [ ]:
print(warehouses["warehouse_name"].unique())

In [ ]:
print("Duplicates in warehouses:", warehouses.duplicated().sum())

In [ ]:
warehouses["warehouse_name"] = warehouses["warehouse_name"].str.strip().str.title()

In [ ]:
warehouses.to_csv("warehouses_clean.csv", index=False)

3. Products

In [ ]:
products = pd.read_csv("D:\\Data_Analysis\\SupplyChain and Inventory Analysis\\products_raw.csv")

In [ ]:
products.info()

In [ ]:
print(products.isnull().sum())
print("Duplicates:", products.duplicated().sum())
print(products["category"].unique())
print(products["sub_category"].unique())
print(products["unit_cost"].describe())

In [ ]:
products = products.drop_duplicates()

In [ ]:
for col in ["category", "sub_category"]:
    products[col] = products[col].str.strip().str.title()
 
products["unit_cost"] = products["unit_cost"].abs()
 
products["unit_cost"] = products.groupby("category")["unit_cost"].transform(
    lambda s: s.fillna(s.median())
)

In [ ]:
before = len(products)
products = products.dropna(subset=["supplier_id"])
print(f"Dropped {before - len(products)} rows with missing supplier_id")
 
assert products["unit_cost"].isnull().sum() == 0
assert (products["unit_cost"] > 0).all()
assert products.duplicated().sum() == 0

In [ ]:
products.to_csv("products_clean.csv", index=False)

4. Inventory Data

In [ ]:
inventory = pd.read_csv("D:\\Data_Analysis\\SupplyChain and Inventory Analysis\\inventory_raw.csv")

In [ ]:
inventory.info()

In [ ]:
print(inventory.isnull().sum())

In [ ]:
print(inventory.duplicated().sum())

In [ ]:
print(inventory["date"].head(10).tolist())   # mixed formats visible here
print(inventory["stock_on_hand"].describe())

In [ ]:
inventory["date"] = pd.to_datetime(inventory["date"], format="mixed", dayfirst=True)

In [ ]:
inventory = inventory.drop_duplicates()

In [ ]:
inventory["stock_on_hand"] = inventory["stock_on_hand"].clip(lower=0)

In [ ]:
inventory = inventory.sort_values(["product_id", "warehouse_id", "date"])

In [ ]:
inventory["stock_on_hand"] = inventory.groupby(["product_id", "warehouse_id"])["stock_on_hand"].transform(
    lambda s: s.ffill().bfill()
)
inventory["reorder_point"] = inventory.groupby(["product_id", "warehouse_id"])["reorder_point"].transform(
    lambda s: s.ffill().bfill()
)

In [ ]:
assert inventory["stock_on_hand"].isnull().sum() == 0
assert (inventory["stock_on_hand"] >= 0).all()
assert inventory.duplicated().sum() == 0

In [ ]:
inventory.to_csv("inventory_clean.csv", index=False)

In [ ]:
inventory.head()

5. Purchase Order Data

In [ ]:
purchases = pd.read_csv("D:\\Data_Analysis\\SupplyChain and Inventory Analysis\\purchase_orders_raw.csv")

In [ ]:
purchases.info()
purchases.describe()

In [ ]:
print(purchases.duplicated().sum())

In [ ]:
print(purchases.isnull().sum())
print("Duplicates:", purchases.duplicated().sum())
print(purchases["status"].unique())
print(purchases["quantity_received"].describe())

In [ ]:
for col in ["order_date", "expected_delivery_date", "actual_delivery_date"]:
    purchases[col] = pd.to_datetime(purchases[col], format="mixed", dayfirst=True)
 
purchases = purchases.drop_duplicates()
 
purchases["status"] = purchases["status"].str.strip().str.title()
 
purchases["quantity_received"] = purchases["quantity_received"].clip(lower=0)
 
mask = purchases["quantity_received"].isnull() & (purchases["status"] != "In Transit")
purchases.loc[mask, "quantity_received"] = purchases.loc[mask, "quantity_ordered"]

In [ ]:

print("Remaining nulls:\\n", purchases.isnull().sum())
 
assert purchases.duplicated().sum() == 0
assert (purchases["quantity_received"].dropna() >= 0).all()


In [ ]:
purchases.to_csv("purchase_orders_clean.csv", index=False)
purchases.head()

6. Sales Orders Data

In [ ]:
sales_orders = pd.read_csv("D:/Data_Analysis/SupplyChain and Inventory Analysis/sales_orders_raw.csv")

In [ ]:
sales_orders.info()
print(sales_orders.isnull().sum())
print("Duplicates:", sales_orders.duplicated().sum())
print(sales_orders["status"].unique())
print((sales_orders["quantity"] == 0).sum(), "zero-quantity orders")

In [ ]:
for col in ["order_date", "ship_date", "delivery_date"]:
    sales_orders[col] = pd.to_datetime(sales_orders[col], format="mixed", dayfirst=True)

In [ ]:
sales_orders = sales_orders.drop_duplicates()
 
sales_orders["status"] = sales_orders["status"].str.strip().str.title()

In [ ]:

before = len(sales_orders)
sales_orders = sales_orders[sales_orders["quantity"] > 0]
print(f"Dropped {before - len(sales_orders)} zero-quantity rows")
 
before = len(sales_orders)
sales_orders = sales_orders.dropna(subset=["customer_id"])
print(f"Dropped {before - len(sales_orders)} rows with missing customer_id")

In [ ]:
assert sales_orders.duplicated().sum() == 0
assert (sales_orders["quantity"] > 0).all()

In [ ]:
sales_orders.to_csv("sales_orders_clean.csv", index=False)
sales_orders.head()

7. Returns Data

In [ ]:
returns = pd.read_csv("d:/Data_Analysis/SupplyChain and Inventory Analysis/returns_raw.csv")

In [ ]:
returns.info()
print(returns.isnull().sum())
print(returns["reason"].unique())

In [ ]:
returns["return_date"] = pd.to_datetime(returns["return_date"], format="mixed", dayfirst=True)
 
returns["reason"] = returns["reason"].str.strip().str.title()
returns["reason"] = returns["reason"].fillna("Unknown")

In [ ]:

assert returns["reason"].isnull().sum() == 0

In [ ]:
returns.to_csv("returns_clean.csv", index=False)
returns.head()

In [ ]:
clean_files = ["suppliers_clean.csv", "warehouses_clean.csv", "products_clean.csv",
               "inventory_clean.csv", "purchase_orders_clean.csv",
               "sales_orders_clean.csv", "returns_clean.csv"]
 
for f in clean_files:
    df = pd.read_csv(f)
    print(f, "→", df.shape, "| nulls:", df.isnull().sum().sum(),
          "| dupes:", df.duplicated().sum())

In [1]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;"
    "DATABASE=SalesData;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string)
)

with engine.connect() as conn:
    print("Connected successfully")

Connected successfully


In [4]:

tables = {
    "suppliers": "suppliers_clean.csv",
    "warehouses": "warehouses_clean.csv",
    "products": "products_clean.csv",
    "inventory": "inventory_clean.csv",
    "purchase_orders": "purchase_orders_clean.csv",
    "sales_orders": "sales_orders_clean.csv",
    "returns": "returns_clean.csv",
}
 
for table_name, file_name in tables.items():
    df = pd.read_csv(file_name)
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
    )
    print(f"Loaded {table_name}: {df.shape[0]} rows")
 

Loaded suppliers: 12 rows
Loaded warehouses: 5 rows
Loaded products: 77 rows
Loaded inventory: 10600 rows
Loaded purchase_orders: 1800 rows
Loaded sales_orders: 5719 rows
Loaded returns: 295 rows
